In [60]:
import pandas as pd
import sqlite3

In [61]:
connection = sqlite3.connect("mieguel_Cervezas.db")


In [62]:
cursor = connection.cursor()

In [63]:
cursor.executescript("""
CREATE TABLE IF NOT EXISTS CERVEZAS (
CodC TEXT PRIMARY KEY,
Envase TEXT,
Capacidad REAL,
Stock INTEGER
);

CREATE TABLE IF NOT EXISTS BARES (
CodB TEXT PRIMARY KEY,
Cif TEXT,
Nombre TEXT,
Localidad TEXT
);

CREATE TABLE IF NOT EXISTS EMPLEADOS (
CodE INTEGER PRIMARY KEY,
Nombre TEXT,
Sueldo REAL
);

CREATE TABLE IF NOT EXISTS REPARTO (
CodE INTEGER,
CodB TEXT,
CodC TEXT,
Fecha TEXT,
Cantidad INTEGER,
FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
FOREIGN KEY (CodB) REFERENCES BARES(CodB),
FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
""")

connection.commit()

In [64]:
cursor.executescript("""
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock) VALUES
('01', 'Botella', 0.2, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1, 288),
('05', 'Barril', 60, 30);

INSERT INTO BARES (CodB, Cif, Nombre, Localidad) VALUES
('001', '11111111X', 'Stop', 'Villa Botijo'),
('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
('003', '-', 'Club Social', 'Las Ranas'),
('004', '33333333Z', 'Otra Ronda', 'La Esponja');

INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo) VALUES
(1, 'Prudencio Caminero', 120000),
(2, 'Vicente Merario', 110000),
(3, 'Valentin Siempre', 100000);

INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad) VALUES
(1, '001', '01', '2005-10-21', 240),
(1, '001', '02', '2005-10-21', 48),
(1, '002', '03', '2005-10-22', 60),
(1, '004', '05', '2005-10-22', 4),
(2, '002', '03', '2005-10-22', 48),
(2, '002', '05', '2005-10-23', 2),
(2, '004', '01', '2005-10-23', 480),
(2, '004', '02', '2005-10-24', 72),
(3, '003', '03', '2005-10-24', 48),
(3, '003', '04', '2005-10-25', 20);
""")

#connection.commit()

IntegrityError: UNIQUE constraint failed: CERVEZAS.CodC

In [ ]:
* Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.


In [68]:
pd.read_sql_query('''
SELECT empleados.Nombre, reparto.Fecha, bares.Nombre 
FROM reparto  
JOIN bares
ON reparto.CodB = bares.CodB
JOIN empleados
ON reparto.CodE = empleados.CodE
WHERE bares.Nombre = "Stop"
AND reparto.Fecha BETWEEN "2005-10-21" AND "2005_10_27";
''', connection)

,Nombre,Fecha,Nombre
0,Prudencio Caminero,2005-10-21,Stop
1,Prudencio Caminero,2005-10-21,Stop


In [ ]:

* Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.


In [76]:
pd.read_sql_query('''
SELECT DISTINCT bares.Cif, bares.Nombre
FROM bares
JOIN reparto
ON bares.CodB = reparto.CodB
JOIN cervezas
ON reparto.CodC = cervezas.CodC
WHERE cervezas.Envase = "Botella"
AND cervezas.Capacidad < 1
ORDER BY bares.localidad
;
''', connection)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [ ]:

* Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.


In [79]:
pd.read_sql_query('''
SELECT bares.Nombre, cervezas.Envase, cervezas.Capacidad, reparto.Fecha, Reparto.Cantidad
FROM reparto
JOIN bares
ON reparto.CodB = bares.codB
JOIN cervezas 
ON reparto.CodC = cervezas.CodC
JOIN empleados
ON reparto.CodE = empleados.CodE
WHERE empleados.nombre = "Prudencio Caminero"
;
''', connection)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4


In [ ]:

* Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.


In [80]:
pd.read_sql_query("""
SELECT DISTINCT B.Nombre
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella'
AND C.Capacidad IN (0.2, 0.33)
""", connection)

,Nombre
0,Stop
1,Otra Ronda


In [ ]:

* Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella.


In [81]:
pd.read_sql_query('''
SELECT BARES.Nombre, EMPLEADOS.Nombre, CERVEZAS.Envase
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
JOIN EMPLEADOS
ON REPARTO.CodE = EMPLEADOS.CodE
WHERE CERVEZAS.Envase = 'Botella'
AND (BARES.Nombre= 'Stop' OR BARES.Nombre = 'Las Vegas')
''',connection)


,Nombre,Nombre,Envase
0,Stop,Prudencio Caminero,Botella
1,Stop,Prudencio Caminero,Botella


In [ ]:

* Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.


In [82]:
pd.read_sql_query('''
SELECT EMPLEADOS.Nombre, COUNT(REPARTO.Fecha), BARES.Localidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN EMPLEADOS
ON REPARTO.CodE = EMPLEADOS.CodE
WHERE BARES.Localidad != 'Villa Botijo'
GROUP BY REPARTO.Fecha
''',connection)



,Nombre,COUNT(REPARTO.Fecha),Localidad
0,Prudencio Caminero,1,La Esponja
1,Vicente Merario,1,La Esponja
2,Vicente Merario,2,La Esponja
3,Valentin Siempre,1,Las Ranas


In [ ]:

* Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.


In [83]:
pd.read_sql_query('''
SELECT BARES.Nombre, BARES.Localidad, CERVEZAS.Capacidad * REPARTO.Cantidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
GROUP BY BARES.Nombre
ORDER BY (CERVEZAS.Capacidad * REPARTO.Cantidad) DESC
LIMIT 1
''',connection)


,Nombre,Localidad,CERVEZAS.Capacidad * REPARTO.Cantidad
0,Otra Ronda,La Esponja,240.0


In [ ]:

* Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.


In [84]:
pd.read_sql_query('''
SELECT BARES.Nombre, CERVEZAS.Envase, CERVEZAS.Capacidad
FROM BARES
JOIN REPARTO
ON REPARTO.CodB = BARES.CodB
JOIN CERVEZAS
ON REPARTO.CodC = CERVEZAS.CodC
WHERE CERVEZAS.Envase = 'Botella'
AND CERVEZAS.Capacidad < 1
GROUP BY BARES.Nombre
''',connection)

,Nombre,Envase,Capacidad
0,Otra Ronda,Botella,0.2
1,Stop,Botella,0.2


In [ ]:

* Subir un 5% el sueldo del empleado que más días haya trabajado.


In [85]:
pd.read_sql_query( '''
SELECT e.CodE, e.Nombre, COUNT(DISTINCT r.Fecha) AS DiasTrabajados
FROM EMPLEADOS AS e
JOIN REPARTO AS r ON e.CodE = r.CodE
GROUP BY e.CodE, e.Nombre
ORDER BY DiasTrabajados DESC
LIMIT 1;''',connection)



,CodE,Nombre,DiasTrabajados
0,2,Vicente Merario,3


In [ ]:

* Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.

In [86]:
cursor.execute('''
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES (
(SELECT CodE FROM EMPLEADOS
WHERE Nombre = 'Vicente Merario'),
(SELECT CodB FROM BARES
WHERE Nombre = 'Stop'),
(SELECT CodC FROM CERVEZAS
WHERE Envase = 'Lata'),
'2005-10-26',
48
);
''')
connection.commit()